# FactIndic 07 – Human evaluation analysis

Reads the filled-in annotation sheets and reports: inter-annotator agreement (Cohen's kappa, target > 0.6), how well each automatic checker detects human-labelled hallucinations (precision, recall, F1), Spearman correlation between automatic scores and human judgements, human hallucination rates per track and input type (a human check of H1 and H2), fluency and coverage ratings, and error types.

| Setting (right-hand panel) | Value |
|---|---|
| Accelerator | **None (CPU)** |
| Internet | **On** |
| Inputs to attach | output of notebook 06 **and** a dataset with the filled-in `human_eval_*_annotator*.xlsx` files |
| Time (full profile) | 2 min |

Run it with **Save Version → Save & Run All (Commit)**. When it finishes, its output (folder `factindic_out`) is
ready to attach to the next notebook. Each notebook's output contains every earlier stage, so you only ever attach
the previous notebook.

In [1]:
# ── Setup: make the factindic code importable ─────────────────────────────────────────
# Kaggle: upload factindic-code.zip as a private dataset and attach it
#         (right-hand panel -> Add Input -> Your Work / Datasets -> factindic-code -> "+").
# Colab:  put factindic-code.zip in My Drive (and run the Google Drive cell first), or upload it to /content.
# Or:     push the repository to GitHub and paste its URL here (needs Internet).
REPO_URL = ""   # e.g. "https://github.com/<your-username>/factindic"

import glob, os, shutil, subprocess, sys, zipfile

ON_COLAB = "google.colab" in sys.modules
SRC = "/content/factindic_src" if ON_COLAB else "/kaggle/working/factindic_src"

def _is_pkg(d):
    return all(os.path.exists(os.path.join(d, f)) for f in ("__init__.py", "pipeline.py", "config.py"))

def _candidates():
    inits = glob.glob("/kaggle/input/**/__init__.py", recursive=True)
    zips = glob.glob("/kaggle/input/**/*.zip", recursive=True)
    if ON_COLAB:  # shallow search only: Google Drive can be huge
        for base in ("/content", "/content/drive/MyDrive"):
            for depth in ("", "*/", "*/*/"):
                inits += glob.glob(f"{base}/{depth}factindic/__init__.py")
            zips += [z for z in glob.glob(f"{base}/*.zip") + glob.glob(f"{base}/*/*.zip")
                     if "factindic" in os.path.basename(z).lower()]
    return sorted(set(inits)), sorted(set(zips))

def _unzip(z):
    try:
        with zipfile.ZipFile(z) as zf:
            hits = [n for n in zf.namelist() if n.endswith("factindic/__init__.py")]
            if hits:
                shutil.rmtree(SRC, ignore_errors=True)
                zf.extractall(SRC)
                return os.path.dirname(os.path.dirname(os.path.join(SRC, sorted(hits, key=len)[-1])))
    except zipfile.BadZipFile:
        pass
    return None

def _find_code():
    inits, zips = _candidates()
    # 0) Colab: always unpack the newest zip from Drive, so replacing the zip there updates the code
    if ON_COLAB:
        for z in sorted(zips, key=os.path.getmtime, reverse=True):
            code = _unzip(z)
            if code:
                print("using", z)
                return code
    # 1) the repository folder (…/factindic/factindic/__init__.py)
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d) and os.path.basename(d) == "factindic":
            return os.path.dirname(d)
    # 2) only the inner package was uploaded (files directly in the dataset): copy it under the right name
    for p in inits:
        d = os.path.dirname(p)
        if _is_pkg(d):
            shutil.copytree(d, os.path.join(SRC, "factindic"), dirs_exist_ok=True)
            for req in glob.glob(os.path.join(os.path.dirname(d), "**", "requirements.txt"), recursive=True)[:1]:
                shutil.copy(req, SRC)
            return SRC
    # 3) a zip that nobody unpacked
    for z in zips:
        code = _unzip(z)
        if code:
            return code
    # 4) running locally from the repository
    for p in ("factindic", "../factindic"):
        if _is_pkg(p):
            return os.path.abspath(os.path.dirname(p) or ".")
    return None

CODE = _find_code()
if CODE is None and REPO_URL:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, SRC], check=True)
    CODE = SRC
if CODE is None:
    where = (sorted(glob.glob("/content/*")) + sorted(glob.glob("/content/drive/MyDrive/*"))[:30]) if ON_COLAB else \
            (sorted(glob.glob("/kaggle/input/*")) + sorted(glob.glob("/kaggle/input/*/*"))[:30])
    how = ("Put factindic-code.zip in My Drive and run the Google Drive cell first (or upload the zip to /content)."
           if ON_COLAB else
           "Attach the dataset: right-hand panel -> Add Input -> Your Work (or Datasets) -> factindic-code -> '+'. "
           "If you uploaded it a minute ago, wait until Kaggle finishes processing it.")
    raise FileNotFoundError("factindic code not found.\n" + how + "\nWhat this notebook can see:\n  "
                            + ("\n  ".join(where) if where else "(nothing)"))
sys.path.insert(0, CODE)
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
req = os.path.join(CODE, "requirements.txt")
pkgs = ["-r", req] if os.path.exists(req) else ["indic-transliteration", "wordfreq", "datasketch", "bert-score",
                                                "python-docx", "feedparser", "trafilatura", "peft", "openpyxl",
                                                "bitsandbytes"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)
import factindic
from factindic import Config
from factindic import pipeline as P
print("factindic", factindic.__version__, "loaded from", CODE)

factindic 1.0.0 loaded from /kaggle/input/datasets/lkr1196/factindic-code/factindic


In [2]:
# ── Configuration ────────────────────────────────────────────────────────────────────
# Use the SAME profile in every notebook of the chain.
#   "full"  – the real experiment
#   "quick" – ~25 articles per language, to check the whole chain on Kaggle first
#   "mock"  – CPU dry run with stand-in models (no GPU, no downloads); never report its numbers
PROFILE = "full"
cfg = Config.make(PROFILE)
print(cfg.to_json())

{
  "profile": "full",
  "seed": 13,
  "langs": [
    "hi",
    "bn"
  ],
  "variants": [
    "mono",
    "cm",
    "roman"
  ],
  "data_source": "xlsum",
  "n_test": 200,
  "n_dev": 30,
  "n_train_cm": 60,
  "min_words": 120,
  "max_words_raw": 1200,
  "lead_words": 300,
  "max_sents": 14,
  "min_summary_words": 8,
  "max_summary_words": 60,
  "near_dup_threshold": 0.8,
  "natural_split": [
    0.7,
    0.1,
    0.2
  ],
  "scrape": true,
  "n_natural_per_lang": 100,
  "backend": "auto",
  "summarizer_model": "Qwen/Qwen3-4B-Instruct-2507",
  "codemix_model": "Qwen/Qwen3-4B-Instruct-2507",
  "openai_base_url": "",
  "openai_model": "",
  "openai_concurrency": 4,
  "max_model_len": 6144,
  "max_batch_tokens": 24000,
  "max_batch_size": 24,
  "repetition_penalty": 1.0,
  "workers": -1,
  "quantize": "auto",
  "summary_max_words": 40,
  "tracks": [
    "A",
    "B",
    "BV"
  ],
  "max_new_tokens_A": 448,
  "max_new_tokens_B": 1536,
  "seq2seq_models": [
    "mt5",
    "indicbart"
  ],
 

In [3]:
out = P.run_stage07(cfg)

Stages available: 01_data, 02_codemix, 03_seq2seq, 04_llm, 05_verify, 06_evaluate
1057 annotated rows from 6 files (LLM judge labels)
# LLM-judge evaluation

The sentence labels below were produced by an LLM judge (unnamed model), not by human annotators. The judge saw each article and its summaries without system, track or input labels.

Labelled sentences: 819

## Detection

| gold | detector | precision | recall | f1 | accuracy | tp | fp | fn | n |
|---|---|---|---|---|---|---|---|---|---|
| Hallucinated | NLI < threshold | 0.159 | 0.167 | 0.162 | 0.836 | 13 | 69 | 65 | 819 |
| Hallucinated | entity/number check | 0.140 | 0.346 | 0.199 | 0.735 | 27 | 166 | 51 | 819 |
| Hallucinated | NLI or entity/number | 0.142 | 0.462 | 0.218 | 0.684 | 36 | 217 | 42 | 819 |
| Hallucinated+Partial | NLI < threshold | 0.451 | 0.141 | 0.215 | 0.670 | 37 | 45 | 225 | 819 |
| Hallucinated+Partial | entity/number check | 0.435 | 0.321 | 0.369 | 0.650 | 84 | 109 | 178 | 819 |
| Hallucinated+Partial | NLI

The final paper draft, now including the human evaluation (download it from the Output tab):

In [4]:
from factindic.paper import build_paper
print(build_paper(cfg.out_root, f"{cfg.out_root}/FactIndic_paper_draft.docx")) 

/kaggle/working/factindic_out/FactIndic_paper_draft.docx


In [5]:
import os
for root, dirs, files in os.walk(cfg.out_root):
    dirs[:] = [d for d in dirs if not d.startswith("_")]
    depth = root[len(cfg.out_root):].count(os.sep)
    if depth <= 1:
        print("  " * depth + os.path.basename(root) + "/", ", ".join(sorted(files))[:300])

factindic_out/ FactIndic_paper_draft.docx
  04_llm/ config.json, env.json, llm_report.md, qg.jsonl, summaries_llm.jsonl
  06_evaluate/ FactIndic_paper_draft.docx, config.json, examples.json, results.md, results_all.csv, results_all.jsonl, summary.md
  02_codemix/ codemix_quality_check.xlsx, codemix_report.md, config.json, env.json, variants.jsonl
  05_verify/ config.json, verification.jsonl, verify_report.md
  03_seq2seq/ config.json, lora_training.json, summaries_seq2seq.jsonl
  07_llm_judge/ codemix_quality.csv, config.json, correlation.csv, detection.csv, error_types.csv, fluency_coverage.csv, judge.json, judge_rates.csv, llm_judge_report.md
  01_data/ config.json, data_report.md, data_stats.csv, docs.jsonl
